nn.Module 是 PyTorch 神经网络的核心基类，它通过自动注册和管理模型参数、子模块及缓冲区，统一封装了前向传播、梯度追踪、设备迁移及序列化等关键功能，是构建一切可训练模型的基石。

# 一 模型定义
以下是通过继承 nn.Module 定义的一个简单的神经网络

In [3]:
import torch
import torch.nn as nn

In [5]:
class NestedNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Sequential(
            nn.Linear(10, 5),
            nn.ReLU(),
        )
        self.layer2 = nn.Linear(5, 2)

    def forward(self, x):
        return self.layer2(self.layer1(x))

model = NestedNet()

# 二 模型参数与结构管理
## 2.1 查看完整模型结构

In [7]:
print(model)

NestedNet(
  (layer1): Sequential(
    (0): Linear(in_features=10, out_features=5, bias=True)
    (1): ReLU()
  )
  (layer2): Linear(in_features=5, out_features=2, bias=True)
)


## 2.2 model.named_modules()

In [45]:
for name, module in model.named_modules():
    print("-" * 10)
    print(f"名称：{name}")
    print(module)

----------
名称：
NestedNet(
  (layer1): Sequential(
    (0): Linear(in_features=10, out_features=5, bias=True)
    (1): ReLU()
  )
  (layer2): Linear(in_features=5, out_features=2, bias=True)
)
----------
名称：layer1
Sequential(
  (0): Linear(in_features=10, out_features=5, bias=True)
  (1): ReLU()
)
----------
名称：layer1.0
Linear(in_features=10, out_features=5, bias=True)
----------
名称：layer1.1
ReLU()
----------
名称：layer2
Linear(in_features=5, out_features=2, bias=True)


## 2.3 model.modules()

In [27]:
for module in model.modules():
    print("-" * 10)
    print(module)

----------
NestedNet(
  (layer1): Sequential(
    (0): Linear(in_features=10, out_features=5, bias=True)
    (1): ReLU()
  )
  (layer2): Linear(in_features=5, out_features=2, bias=True)
)
----------
Sequential(
  (0): Linear(in_features=10, out_features=5, bias=True)
  (1): ReLU()
)
----------
Linear(in_features=10, out_features=5, bias=True)
----------
ReLU()
----------
Linear(in_features=5, out_features=2, bias=True)


## 2.4 model.named_children()

In [44]:
for name, children in model.named_children():
    print("-" * 10)
    print(f"名称：{name}")
    print(f"类型：{type(children)}")
    print(children)

----------
名称：layer1
类型：<class 'torch.nn.modules.container.Sequential'>
Sequential(
  (0): Linear(in_features=10, out_features=5, bias=True)
  (1): ReLU()
)
----------
名称：layer2
类型：<class 'torch.nn.modules.linear.Linear'>
Linear(in_features=5, out_features=2, bias=True)


## 2.5 model.children()

In [43]:
for children in model.children():
    print("-" * 10)
    print(f"类型：{type(children)}")
    print(children)

----------
类型：<class 'torch.nn.modules.container.Sequential'>
Sequential(
  (0): Linear(in_features=10, out_features=5, bias=True)
  (1): ReLU()
)
----------
类型：<class 'torch.nn.modules.linear.Linear'>
Linear(in_features=5, out_features=2, bias=True)


## 2.6 model.named_parameters()

In [41]:
for name, param in model.named_parameters():
    print("-" * 10)
    print(f"名称：{name}")
    print(f"形状：{param.shape}")
    print(f"类型：{type(param)}")
    print(f"参数：{param}")

----------
名称：layer1.0.weight
形状：torch.Size([5, 10])
类型：<class 'torch.nn.parameter.Parameter'>
参数：Parameter containing:
tensor([[ 0.0040, -0.3133,  0.0496, -0.1655,  0.0018, -0.1061,  0.0215,  0.1571,
          0.1679,  0.0162],
        [ 0.0432,  0.0318, -0.2239, -0.2245, -0.0265,  0.2381,  0.2624,  0.0201,
          0.1175,  0.2822],
        [-0.0905, -0.2379, -0.1528,  0.0276,  0.0425,  0.2974, -0.2502, -0.0871,
          0.2139,  0.2570],
        [ 0.0084, -0.0483, -0.0151,  0.0681,  0.2400, -0.2084,  0.1236, -0.2361,
         -0.2335, -0.0027],
        [-0.2576,  0.1868, -0.2214,  0.1283,  0.0030,  0.0026,  0.2483,  0.1630,
         -0.1236,  0.2112]], requires_grad=True)
----------
名称：layer1.0.bias
形状：torch.Size([5])
类型：<class 'torch.nn.parameter.Parameter'>
参数：Parameter containing:
tensor([-0.2072,  0.1794,  0.0332,  0.0391,  0.1966], requires_grad=True)
----------
名称：layer2.weight
形状：torch.Size([2, 5])
类型：<class 'torch.nn.parameter.Parameter'>
参数：Parameter containing:
tensor([[

## 2.7 model.parameters()

In [42]:
for param in model.parameters():
    print("-" * 10)
    print(f"形状：{param.shape}")
    print(f"类型：{type(param)}")
    print(f"参数：{param}")

----------
形状：torch.Size([5, 10])
类型：<class 'torch.nn.parameter.Parameter'>
参数：Parameter containing:
tensor([[ 0.0040, -0.3133,  0.0496, -0.1655,  0.0018, -0.1061,  0.0215,  0.1571,
          0.1679,  0.0162],
        [ 0.0432,  0.0318, -0.2239, -0.2245, -0.0265,  0.2381,  0.2624,  0.0201,
          0.1175,  0.2822],
        [-0.0905, -0.2379, -0.1528,  0.0276,  0.0425,  0.2974, -0.2502, -0.0871,
          0.2139,  0.2570],
        [ 0.0084, -0.0483, -0.0151,  0.0681,  0.2400, -0.2084,  0.1236, -0.2361,
         -0.2335, -0.0027],
        [-0.2576,  0.1868, -0.2214,  0.1283,  0.0030,  0.0026,  0.2483,  0.1630,
         -0.1236,  0.2112]], requires_grad=True)
----------
形状：torch.Size([5])
类型：<class 'torch.nn.parameter.Parameter'>
参数：Parameter containing:
tensor([-0.2072,  0.1794,  0.0332,  0.0391,  0.1966], requires_grad=True)
----------
形状：torch.Size([2, 5])
类型：<class 'torch.nn.parameter.Parameter'>
参数：Parameter containing:
tensor([[-0.2064, -0.1170,  0.0831, -0.2932,  0.0230],
       

## 2.8 model.state_dict()

In [28]:
model.state_dict()

OrderedDict([('layer1.0.weight',
              tensor([[ 0.0040, -0.3133,  0.0496, -0.1655,  0.0018, -0.1061,  0.0215,  0.1571,
                        0.1679,  0.0162],
                      [ 0.0432,  0.0318, -0.2239, -0.2245, -0.0265,  0.2381,  0.2624,  0.0201,
                        0.1175,  0.2822],
                      [-0.0905, -0.2379, -0.1528,  0.0276,  0.0425,  0.2974, -0.2502, -0.0871,
                        0.2139,  0.2570],
                      [ 0.0084, -0.0483, -0.0151,  0.0681,  0.2400, -0.2084,  0.1236, -0.2361,
                       -0.2335, -0.0027],
                      [-0.2576,  0.1868, -0.2214,  0.1283,  0.0030,  0.0026,  0.2483,  0.1630,
                       -0.1236,  0.2112]])),
             ('layer1.0.bias',
              tensor([-0.2072,  0.1794,  0.0332,  0.0391,  0.1966])),
             ('layer2.weight',
              tensor([[-0.2064, -0.1170,  0.0831, -0.2932,  0.0230],
                      [ 0.0199, -0.1906,  0.4012,  0.1831,  0.3648]])),
      

# 三 模型权重保存与加载

## 3.1 权重保存

In [29]:
torch.save(model.state_dict(), 'model.pth')

## 3.2 权重加载

In [30]:
model_new = NestedNet()
model_new.state_dict()

OrderedDict([('layer1.0.weight',
              tensor([[ 0.2425, -0.2177,  0.2791,  0.3104,  0.0886, -0.1655,  0.2893, -0.2757,
                        0.2808,  0.0083],
                      [-0.2324, -0.3072, -0.0960,  0.1098,  0.2009, -0.2215,  0.0177, -0.2997,
                        0.0767,  0.2587],
                      [ 0.0924, -0.2446, -0.1323, -0.1091,  0.0628,  0.2788, -0.2785, -0.1565,
                       -0.1210, -0.2236],
                      [ 0.1073,  0.1780, -0.2954, -0.0520, -0.0688,  0.2951,  0.0650, -0.1015,
                        0.2531,  0.2323],
                      [ 0.0312, -0.2798,  0.0579,  0.0630,  0.1928, -0.1917,  0.1055,  0.1701,
                       -0.2907, -0.1218]])),
             ('layer1.0.bias',
              tensor([ 0.2354,  0.0909, -0.0635, -0.2809, -0.0988])),
             ('layer2.weight',
              tensor([[-0.2110, -0.1838,  0.4121,  0.1049, -0.3718],
                      [-0.2184,  0.0953,  0.0325,  0.3450,  0.0331]])),
      

In [47]:
model_new.load_state_dict(torch.load(
    'model.pth',
    map_location=torch.device('cpu')
))
model_new.state_dict()

OrderedDict([('layer1.0.weight',
              tensor([[ 0.0040, -0.3133,  0.0496, -0.1655,  0.0018, -0.1061,  0.0215,  0.1571,
                        0.1679,  0.0162],
                      [ 0.0432,  0.0318, -0.2239, -0.2245, -0.0265,  0.2381,  0.2624,  0.0201,
                        0.1175,  0.2822],
                      [-0.0905, -0.2379, -0.1528,  0.0276,  0.0425,  0.2974, -0.2502, -0.0871,
                        0.2139,  0.2570],
                      [ 0.0084, -0.0483, -0.0151,  0.0681,  0.2400, -0.2084,  0.1236, -0.2361,
                       -0.2335, -0.0027],
                      [-0.2576,  0.1868, -0.2214,  0.1283,  0.0030,  0.0026,  0.2483,  0.1630,
                       -0.1236,  0.2112]])),
             ('layer1.0.bias',
              tensor([-0.2072,  0.1794,  0.0332,  0.0391,  0.1966])),
             ('layer2.weight',
              tensor([[-0.2064, -0.1170,  0.0831, -0.2932,  0.0230],
                      [ 0.0199, -0.1906,  0.4012,  0.1831,  0.3648]])),
      